# Phase 7: TabPFN, LightGBM & Stacking

Adds a tabular **foundation model** and a third **GBDT**, then combines the strongest
learners into an ensemble. Same 5-fold Stratified CV as Phase 2/6.

- **LightGBM** — fast histogram-based gradient boosting (completes the GBDT trio with XGBoost + CatBoost).
- **TabPFN** — a transformer pre-trained on synthetic tables; predicts in one forward pass, no tuning. Paper: https://arxiv.org/abs/2207.01848
- **Stacking / Blending** — average and a Logistic-Regression meta-learner over the
  out-of-fold predictions of TabPFN + CatBoost + LightGBM + XGBoost.


In [ ]:
# Install if needed:
# %pip install tabpfn lightgbm catboost xgboost scikit-learn
# Note: TabPFN downloads pretrained weights on the first .fit() (needs internet once).

In [ ]:
import warnings, time
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from tabpfn import TabPFNClassifier

SEED = 42
np.random.seed(SEED)

## Step 0: Read the cleaned data

In [ ]:
from pathlib import Path

DATA_DIR = Path("../data/processed")
if not DATA_DIR.exists():
    DATA_DIR = Path("data/processed")

train = pd.read_csv(DATA_DIR / "train_cleaned.csv")
TARGET = "Class/ASD"
X = train.drop(columns=[TARGET])
y = train[TARGET]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
X_np, y_np = X.values.astype(np.float32), y.values.astype(np.int64)
print(f"train: {X.shape} | classes: {y.value_counts().to_dict()}")

leaderboard, oof_preds = [], {}   # results + out-of-fold probs (reused for stacking)


def _row(name, acc, f1, auc):
    return {"Model": name,
            "Accuracy": np.mean(acc), "Accuracy-Std": np.std(acc),
            "F1-Score": np.mean(f1), "F1-Std": np.std(f1),
            "ROC-AUC": np.mean(auc), "ROC-AUC-Std": np.std(auc)}

def get_oof(make_model, name):
    """Out-of-fold probabilities + per-fold CV metrics for one model."""
    oof = np.zeros(len(y_np)); auc, f1, acc = [], [], []
    for tr, va in cv.split(X_np, y_np):
        m = make_model(); m.fit(X_np[tr], y_np[tr])
        p = m.predict_proba(X_np[va])[:, 1]
        oof[va] = p; pred = (p >= 0.5).astype(int)
        auc.append(roc_auc_score(y_np[va], p))
        f1.append(f1_score(y_np[va], pred)); acc.append(accuracy_score(y_np[va], pred))
    return oof, _row(name, acc, f1, auc)

def score_oof(oof, name):
    """Per-fold CV metrics from an existing OOF prob vector (for ensembles)."""
    auc, f1, acc = [], [], []
    for _, va in cv.split(X_np, y_np):
        pred = (oof[va] >= 0.5).astype(int)
        auc.append(roc_auc_score(y_np[va], oof[va]))
        f1.append(f1_score(y_np[va], pred)); acc.append(accuracy_score(y_np[va], pred))
    return _row(name, acc, f1, auc)

### Step 1: LightGBM (5-fold Stratified CV)

In [ ]:
def make_lgbm():
    return LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31,
                          max_depth=6, min_child_samples=20, subsample=0.8,
                          colsample_bytree=0.8, reg_lambda=1.0,
                          class_weight="balanced", random_state=SEED, verbosity=-1)

t0 = time.time()
oof_preds["LightGBM"], row = get_oof(make_lgbm, "LightGBM")
leaderboard.append(row)
print(f"LightGBM | AUC {row['ROC-AUC']:.4f} +/- {row['ROC-AUC-Std']:.4f} "
      f"| F1 {row['F1-Score']:.4f} | Acc {row['Accuracy']:.4f} | {time.time()-t0:.1f}s")

### Step 2: TabPFN (foundation model)

`ignore_pretraining_limits=True` allows > 1000 rows. Runs on CPU here, so expect a few
minutes for the 5 folds; the first fit also downloads the pretrained weights.

In [ ]:
def make_tabpfn():
    return TabPFNClassifier(device="cpu", ignore_pretraining_limits=True, random_state=SEED)

t0 = time.time()
oof_preds["TabPFN"], row = get_oof(make_tabpfn, "TabPFN")
leaderboard.append(row)
print(f"TabPFN | AUC {row['ROC-AUC']:.4f} +/- {row['ROC-AUC-Std']:.4f} "
      f"| F1 {row['F1-Score']:.4f} | Acc {row['Accuracy']:.4f} | {time.time()-t0:.1f}s")

### Step 3: Stacking & Blending

Base learners: TabPFN + CatBoost + LightGBM + XGBoost (LightGBM/TabPFN OOF reused).
Combine their out-of-fold probabilities two ways: a simple average, and a Logistic
Regression meta-learner trained on them.

In [ ]:
spw = (y_np == 0).sum() / (y_np == 1).sum()   # scale_pos_weight for XGBoost
make_cat = lambda: CatBoostClassifier(iterations=400, learning_rate=0.05, depth=6,
                                      l2_leaf_reg=3.0, auto_class_weights="Balanced",
                                      random_seed=SEED, verbose=False)
make_xgb = lambda: XGBClassifier(n_estimators=400, learning_rate=0.05, max_depth=5,
                                 subsample=0.8, colsample_bytree=0.8, eval_metric="logloss",
                                 scale_pos_weight=spw, random_state=SEED)

# OOF for the remaining base learners (LightGBM, TabPFN already done above)
for name, make in [("CatBoost", make_cat), ("XGBoost", make_xgb)]:
    if name not in oof_preds:
        oof_preds[name], _ = get_oof(make, name)

base = ["TabPFN", "CatBoost", "LightGBM", "XGBoost"]
meta = np.column_stack([oof_preds[m] for m in base])

# (a) simple average blend
blend_oof = meta.mean(axis=1)
leaderboard.append(score_oof(blend_oof, "Blend (avg of 4)"))

# (b) stacking: Logistic Regression meta-learner on OOF predictions
stack_oof = np.zeros(len(y_np))
for tr, va in cv.split(meta, y_np):
    lr = LogisticRegression(max_iter=1000, random_state=SEED)
    lr.fit(meta[tr], y_np[tr])
    stack_oof[va] = lr.predict_proba(meta[va])[:, 1]
leaderboard.append(score_oof(stack_oof, "Stacking (LR)"))

print("Base models:", base)
print(f"Blend    AUC: {roc_auc_score(y_np, blend_oof):.4f}")
print(f"Stacking AUC: {roc_auc_score(y_np, stack_oof):.4f}")

### Step 4: Leaderboard

In [ ]:
results_df = (pd.DataFrame(leaderboard).round(4)
              .sort_values("ROC-AUC", ascending=False).reset_index(drop=True))
print("\n--- LEADERBOARD ---")
display(results_df)